In [1]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

In [2]:
df = pd.read_csv("../dataset/fake_job_postings.csv")

print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

Dataset shape: (17880, 18)
Columns:
['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']


In [3]:
target = df["fraudulent"]

features = df.drop(
    columns=["job_id", "salary_range", "fraudulent"]
)

print("Features shape:", features.shape)
print("Target shape:", target.shape)
print("\nTarget distribution:")
print(target.value_counts())

Features shape: (17880, 15)
Target shape: (17880,)

Target distribution:
fraudulent
0    17014
1      866
Name: count, dtype: int64


In [4]:
import re

text_columns = [
    "title",
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

df["combined_text"] = (
    df[text_columns]
    .fillna("")
    .agg(" ".join, axis=1)
)

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["combined_text"].apply(clean_text)

print(df[["combined_text", "clean_text"]].head())

                                       combined_text  \
0  Marketing Intern We're Food52, and we've creat...   
1  Customer Service - Cloud Video Production 90 S...   
2  Commissioning Machinery Assistant (CMA) Valor ...   
3  Account Executive - Washington DC Our passion ...   
4  Bill Review Manager SpotSource Solutions LLC i...   

                                          clean_text  
0  marketing intern we re food and we ve created ...  
1  customer service cloud video production second...  
2  commissioning machinery assistant cma valor se...  
3  account executive washington dc our passion fo...  
4  bill review manager spotsource solutions llc i...  


In [5]:
categorical_columns = [
    "location",
    "department",
    "employment_type",
    "required_experience",
    "required_education",
    "industry",
    "function"
]

numeric_columns = [
    "telecommuting",
    "has_company_logo",
    "has_questions"
]

features = df[
    ["clean_text"] + categorical_columns + numeric_columns
]

print("Feature columns:")
print(features.columns.tolist())

print("\nFeatures shape:", features.shape)

Feature columns:
['clean_text', 'location', 'department', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'telecommuting', 'has_company_logo', 'has_questions']

Features shape: (17880, 11)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.20,
    random_state=42,
    stratify=target
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

X_train: (14304, 11)
X_test : (3576, 11)
y_train: (14304,)
y_test : (3576,)

Training target distribution:
fraudulent
0    13611
1      693
Name: count, dtype: int64

Testing target distribution:
fraudulent
0    3403
1     173
Name: count, dtype: int64


In [7]:
preprocessor = joblib.load("../models/preprocessor.pkl")

print("Preprocessor loaded successfully.")
print(preprocessor)

Preprocessor loaded successfully.
ColumnTransformer(transformers=[('text',
                                 TfidfVectorizer(max_features=10000,
                                                 ngram_range=(1, 2),
                                                 stop_words='english'),
                                 'clean_text'),
                                ('categorical',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['location', 'department', 'employment_type',
                                  'required_experience', 'required_education',
                                  'industry', 'function']),
                                ('numeric', 'passthrough',
                                 ['telecommuting', 'has_company_logo',
                                  'has_questions'])])


In [8]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape :", X_test_processed.shape)

Processed training shape: (14304, 14114)
Processed testing shape : (3576, 14114)


In [9]:
logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_processed, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


In [10]:
y_pred = logistic_model.predict(X_test_processed)

y_prob = logistic_model.predict_proba(X_test_processed)[:, 1]

print("Predictions generated successfully.")
print("Number of predictions:", len(y_pred))

Predictions generated successfully.
Number of predictions: 3576


In [12]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Logistic Regression Evaluation")
print("--------------------------------")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")

Logistic Regression Evaluation
--------------------------------
Accuracy : 0.9760
Precision: 0.6900
Recall   : 0.9133
F1-Score : 0.7861
ROC-AUC  : 0.9885


In [13]:
print("Classification Report")
print("=====================")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Legitimate", "Fraudulent"]
    )
)

Classification Report
              precision    recall  f1-score   support

  Legitimate       1.00      0.98      0.99      3403
  Fraudulent       0.69      0.91      0.79       173

    accuracy                           0.98      3576
   macro avg       0.84      0.95      0.89      3576
weighted avg       0.98      0.98      0.98      3576



In [14]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix")
print("================")
print(cm)

Confusion Matrix
[[3332   71]
 [  15  158]]


In [16]:
joblib.dump(
    logistic_model,
    "../models/logistic_regression.pkl"
)

print("Logistic Regression model saved successfully.")

Logistic Regression model saved successfully.


In [17]:
results = {
    "Model": "Logistic Regression",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1,
    "ROC-AUC": roc_auc
}

results_df = pd.DataFrame([results])

results_df

,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,Logistic Regression,0.975951,0.689956,0.913295,0.78607,0.988492
